In [1]:
import pandas as pd
import json

In [2]:
from pathlib import Path

pathlist = Path("logs").rglob('*.log')
for path in pathlist:
    print(path)

logs\Participant1_task1_2026-03-19_10-39-42.247066.log
logs\Participant1_task1_2026-03-19_10-52-04.778229.log
logs\Participant2_task1_2026-03-19_10-47-08.454254.log
logs\Participant2_task1_2026-03-19_10-53-52.696965.log
logs\Participant3_task1_2026-03-19_10-48-46.985972.log
logs\Participant3_task1_2026-03-19_10-55-45.182661.log


In [3]:
from pathlib import Path

list_of_logs = []

pathlist = Path("logs").rglob('*.log')
for path in pathlist:
    uid = str(path).split("_")[0].split("\\")[1]
    with open(path) as f:
        data = f.read().splitlines()

    data_dicts = [json.loads(line) for line in data]
    logs = pd.DataFrame(data_dicts)
    logs["uid"] = uid
    list_of_logs.append(logs)

data = pd.concat(list_of_logs)
data["timestamp"]= pd.to_datetime(data["timestamp"])

In [4]:
data.to_csv("data/raw_logs_compiled.csv", index=False)
data

,type,timestamp,sessionID,uid,query,docid,rank,page,url,windowLocation,fromURL,toURL,clicked,fromPage,toPage
0,TaskStarted,2026-03-19 10:35:27.215000+00:00,0b0cd4e1-3bd5-41fc-9e24-d7a34709df8c,Participant1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,idSubmitted,2026-03-19 10:35:27.220000+00:00,0b0cd4e1-3bd5-41fc-9e24-d7a34709df8c,Participant1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,queryBoxFocused,2026-03-19 10:35:28.366000+00:00,0b0cd4e1-3bd5-41fc-9e24-d7a34709df8c,Participant1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,querySubmitted,2026-03-19 10:35:36.606000+00:00,0b0cd4e1-3bd5-41fc-9e24-d7a34709df8c,Participant1,what does the fox say,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,searchResultGenerated,2026-03-19 10:35:38.238000+00:00,0b0cd4e1-3bd5-41fc-9e24-d7a34709df8c,Participant1,what does the fox say,2891,1,1,https://www.africanstorybook.org/#,http://localhost:7001/result?query=what+does+t...,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,cursorEnteredSnippet,2026-03-19 10:55:38.828000+00:00,bc54211a-9aac-4d27-96b4-9efdc0db6ace,Participant3,clime change causes,7349,9,1,http://www.gutenberg.org/files/38750/38750-h/3...,http://localhost:7001/result?query=clime+chang...,NaN,NaN,NaN,NaN,NaN
84,cursorLeftSnippet,2026-03-19 10:55:38.944000+00:00,bc54211a-9aac-4d27-96b4-9efdc0db6ace,Participant3,clime change causes,7349,9,1,http://www.gutenberg.org/files/38750/38750-h/3...,http://localhost:7001/result?query=clime+chang...,NaN,NaN,NaN,NaN,NaN
85,wentBack,2026-03-19 10:55:43.552000+00:00,bc54211a-9aac-4d27-96b4-9efdc0db6ace,Participant3,clime change causes,NaN,NaN,NaN,NaN,NaN,https://kids.frontiersin.org/article/10.3389/f...,http://localhost:7001/result?query=clime+chang...,NaN,NaN,NaN
86,ClickedEndTask,2026-03-19 10:55:44.239000+00:00,bc54211a-9aac-4d27-96b4-9efdc0db6ace,Participant3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
data.iloc[1]["timestamp"] - data.iloc[0]["timestamp"] 

Timedelta('0 days 00:00:00.005000')

In [6]:
results_data = data.loc[data["type"]=="searchResultGenerated"][["uid", "sessionID", "timestamp", "query", "docid", "url", "rank", "page"]]
results_data["retriever_rank"] = [int(rank) + (int(page)-1)*10 for (rank, page) in zip(results_data["rank"], results_data["page"])]
results_data = results_data.rename({"rank":"SERP_rank", "page": "SERP_page"}, axis=1)
results_data.drop_duplicates(inplace=True)

In [7]:
results_data.to_csv("data/results_data.csv", index=False)
results_data

,uid,sessionID,timestamp,query,docid,url,SERP_rank,SERP_page,retriever_rank
4,Participant1,0b0cd4e1-3bd5-41fc-9e24-d7a34709df8c,2026-03-19 10:35:38.238000+00:00,what does the fox say,2891,https://www.africanstorybook.org/#,1,1,1
5,Participant1,0b0cd4e1-3bd5-41fc-9e24-d7a34709df8c,2026-03-19 10:35:38.238000+00:00,what does the fox say,7175,http://www.gutenberg.org/cache/epub/3152/pg315...,2,1,2
6,Participant1,0b0cd4e1-3bd5-41fc-9e24-d7a34709df8c,2026-03-19 10:35:38.239000+00:00,what does the fox say,7034,http://www.gutenberg.org/files/25359/25359-h/2...,3,1,3
7,Participant1,0b0cd4e1-3bd5-41fc-9e24-d7a34709df8c,2026-03-19 10:35:38.239000+00:00,what does the fox say,5743,http://www.gutenberg.org/files/24479/24479-h/2...,4,1,4
8,Participant1,0b0cd4e1-3bd5-41fc-9e24-d7a34709df8c,2026-03-19 10:35:38.239000+00:00,what does the fox say,5489,http://www.gutenberg.org/ebooks/19399,5,1,5
...,...,...,...,...,...,...,...,...,...
63,Participant3,bc54211a-9aac-4d27-96b4-9efdc0db6ace,2026-03-19 10:55:27.949000+00:00,clime change causes,2994,https://kids.frontiersin.org/article/10.3389/f...,6,1,6
64,Participant3,bc54211a-9aac-4d27-96b4-9efdc0db6ace,2026-03-19 10:55:27.949000+00:00,clime change causes,2915,https://kids.frontiersin.org/article/10.3389/f...,7,1,7
65,Participant3,bc54211a-9aac-4d27-96b4-9efdc0db6ace,2026-03-19 10:55:27.950000+00:00,clime change causes,2638,https://www.africanstorybook.org/#,8,1,8
66,Participant3,bc54211a-9aac-4d27-96b4-9efdc0db6ace,2026-03-19 10:55:27.951000+00:00,clime change causes,7349,http://www.gutenberg.org/files/38750/38750-h/3...,9,1,9


In [8]:
clicked_results_data = data.loc[data["type"]=="clickedResult"][["uid", "sessionID", "timestamp", "query", "docid", "url", "rank", "page"]]
clicked_results_data = clicked_results_data.rename({"page": "SERP_page", "rank": "SERP_rank"}, axis=1)
clicked_results_data.drop_duplicates(inplace=True)

In [9]:
clicked_results_data.to_csv("data/clicked_results_data.csv", index=False)
clicked_results_data

,uid,sessionID,timestamp,query,docid,url,SERP_rank,SERP_page
21,Participant1,0b0cd4e1-3bd5-41fc-9e24-d7a34709df8c,2026-03-19 10:35:47.780000+00:00,what does the fox say,7163,http://www.gutenberg.org/cache/epub/3152/pg315...,8,1
25,Participant1,0b0cd4e1-3bd5-41fc-9e24-d7a34709df8c,2026-03-19 10:35:57.998000+00:00,what does the fox say,7175,http://www.gutenberg.org/cache/epub/3152/pg315...,2,1
40,Participant1,0b0cd4e1-3bd5-41fc-9e24-d7a34709df8c,2026-03-19 10:36:09.096000+00:00,what does the fox say,7021,http://www.gutenberg.org/files/25359/25359-h/2...,3,3
43,Participant1,0b0cd4e1-3bd5-41fc-9e24-d7a34709df8c,2026-03-19 10:36:25.235000+00:00,what does the fox say,6735,http://www.gutenberg.org/files/17767/17767-h/1...,9,3
57,Participant1,0b0cd4e1-3bd5-41fc-9e24-d7a34709df8c,2026-03-19 10:36:46.058000+00:00,what does the fox say,6725,http://www.gutenberg.org/files/19522/19522-h/1...,1,4
...,...,...,...,...,...,...,...,...
25,Participant3,bc54211a-9aac-4d27-96b4-9efdc0db6ace,2026-03-19 10:54:27.344000+00:00,climate change greenhouse,1960,https://simple.wikipedia.org/wiki/Climate,3,1
35,Participant3,bc54211a-9aac-4d27-96b4-9efdc0db6ace,2026-03-19 10:54:39.805000+00:00,climate change greenhouse,2599,https://kids.frontiersin.org/article/10.3389/f...,10,1
52,Participant3,bc54211a-9aac-4d27-96b4-9efdc0db6ace,2026-03-19 10:54:57.391000+00:00,climate change greenhouse,4963,http://www.gutenberg.org/files/15417/15417-h/1...,2,2
73,Participant3,bc54211a-9aac-4d27-96b4-9efdc0db6ace,2026-03-19 10:55:31.885000+00:00,clime change causes,7349,http://www.gutenberg.org/files/38750/38750-h/3...,9,1


In [10]:
session_data_rows = []

for uid in data["uid"].unique():
    user_sessions = data.loc[data["uid"]==uid]
    for session_id in user_sessions["sessionID"].unique():
        temp = user_sessions.loc[user_sessions["sessionID"]==session_id]
        start_time = temp.loc[temp["type"]=="TaskStarted"]["timestamp"].values[0]
        end_time = temp.loc[temp["type"]=="TaskEndConfirmed"]["timestamp"].values[0]
        session_data_rows.append([uid, session_id, start_time, end_time])

session_data = pd.DataFrame(session_data_rows, columns=["uid", "sessionID", "sessionStartTime", "sessionEndTime"])

In [11]:
session_data.to_csv("data/session_data.csv", index=False)
session_data

,uid,sessionID,sessionStartTime,sessionEndTime
0,Participant1,0b0cd4e1-3bd5-41fc-9e24-d7a34709df8c,2026-03-19 10:35:27.215,2026-03-19 10:39:42.105
1,Participant1,ac19467f-cd75-43ee-b0bb-d4a19dad75d0,2026-03-19 10:49:08.166,2026-03-19 10:52:04.684
2,Participant2,2db4a128-a7b0-462d-9432-0f66ce1c7a9f,2026-03-19 10:42:45.050,2026-03-19 10:47:08.329
3,Participant2,358665ed-05f9-463c-b357-29b94091e948,2026-03-19 10:52:24.510,2026-03-19 10:53:52.655
4,Participant3,5cb5b8f2-6892-4106-9aaa-2be16016a862,2026-03-19 10:47:15.477,2026-03-19 10:48:46.952
5,Participant3,bc54211a-9aac-4d27-96b4-9efdc0db6ace,2026-03-19 10:54:01.075,2026-03-19 10:55:45.138
